In [3]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================
# LOAD DATASET
# ==========================================

housing = fetch_california_housing(as_frame=True)

df = housing.frame

print("Dataset Shape:", df.shape)

# ==========================================
# FEATURES & TARGET
# ==========================================

X = df.drop("MedHouseVal", axis=1)

y = df["MedHouseVal"]

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# HYPERPARAMETER SEARCH FUNCTION
# ==========================================

def find_best_random_forest(
    X_train,
    y_train
):

    param_grid = {

        "n_estimators": [100],

        "max_depth": [10, 15],

        "min_samples_split": [2, 5],

        "min_samples_leaf": [1, 5],

        "max_features": ["sqrt", "log2"]

    }

    rf = RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    )

    grid = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    return grid.best_estimator_, grid.best_params_, grid.best_score_

# ==========================================
# FIND BEST MODEL
# ==========================================

best_model, best_params, best_cv_score = find_best_random_forest(
    X_train,
    y_train
)

print("\nBest Parameters")
print(best_params)

print("\nBest CV Score")
print(round(best_cv_score, 4))

# ==========================================
# TRAIN BEST MODEL
# ==========================================

best_model.fit(X_train, y_train)

# ==========================================
# PREDICTIONS
# ==========================================

y_train_pred = best_model.predict(X_train)

y_test_pred = best_model.predict(X_test)

# ==========================================
# EVALUATION
# ==========================================

train_r2 = r2_score(y_train, y_train_pred)

test_r2 = r2_score(y_test, y_test_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

mae = mean_absolute_error(
    y_test,
    y_test_pred
)

print("\n" + "="*50)

print("FINAL MODEL RESULTS")

print("="*50)

print(f"Train R² : {train_r2:.4f}")

print(f"Test R²  : {test_r2:.4f}")

print(f"RMSE      : {rmse:.4f}")

print(f"MAE       : {mae:.4f}")

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

feature_importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": best_model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")

print(feature_importance)

Dataset Shape: (20640, 9)
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best Parameters
{'max_depth': 15, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best CV Score
0.8118

FINAL MODEL RESULTS
Train R² : 0.9501
Test R²  : 0.8066
RMSE      : 0.5035
MAE       : 0.3327

Feature Importance
      Feature  Importance
0      MedInc    0.412882
5    AveOccup    0.127444
6    Latitude    0.121342
7   Longitude    0.116127
2    AveRooms    0.105999
1    HouseAge    0.052180
3   AveBedrms    0.036758
4  Population    0.027268
